In [1]:
!pip install -q langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os

os.makedirs("documents", exist_ok=True)

policies = {
    "HR_Policy.txt": """
HR Policy

Working hours are from 9:00 AM to 6:00 PM, Monday to Friday.
Employees must maintain professional behavior in the workplace.
New employees have a probation period of six months.
Employee performance is reviewed annually.
Employees can contact the HR department for policy clarification.
""",

    "Leave_Policy.txt": """
Leave Policy

Employees are entitled to 12 casual leave days per year.
Employees are entitled to 10 sick leave days per year.
Employees are entitled to 15 annual leave days per year.
Leave requests must be submitted through the approved leave management system.
Manager approval is required before taking planned leave.
In case of emergency leave, the employee should notify the manager as soon as possible.
""",

    "Attendance_Policy.txt": """
Attendance Policy

Employees are expected to maintain regular attendance.
Employees should inform their manager if they expect to arrive late.
Attendance must be recorded using the approved attendance system.
Unplanned absence must follow the company's leave procedure.
Repeated attendance issues may be reviewed by management.
""",

    "WFH_Policy.txt": """
Work From Home Policy

Eligible employees can work from home up to 2 days per week.
Work from home requires manager approval.
Work from home arrangements depend on business requirements.
Employees must follow normal working hours while working from home.
Employees must remain available through approved communication tools.
Employees may be required to attend office meetings, training, or team activities.
""",

    "Reimbursement_Policy.txt": """
Reimbursement Policy

Approved business expenses are eligible for reimbursement.
Employees must submit valid receipts for reimbursement.
Reimbursement requests should normally be submitted within 30 days.
Manager and Finance approval may be required.
Approved business travel expenses can be reimbursed.
Personal expenses are not eligible for reimbursement.
""",

    "Employee_Guidelines.txt": """
Employee Guidelines

Employees should communicate professionally with colleagues.
Employees must protect confidential company information.
Employees should use strong passwords and must never share passwords.
Company resources should mainly be used for authorized business purposes.
Security incidents and policy violations should be reported through proper channels.
"""
}

for filename, content in policies.items():
    with open("documents/" + filename, "w") as f:
        f.write(content)

print("Policy documents created successfully!")

Policy documents created successfully!


In [3]:
from langchain_community.document_loaders import TextLoader

documents = []

for filename in os.listdir("documents"):
    if filename.endswith(".txt"):
        filepath = os.path.join("documents", filename)

        loader = TextLoader(filepath, encoding="utf-8")
        docs = loader.load()

        for doc in docs:
            doc.metadata["source"] = filename
            doc.metadata["department"] = "Human Resources"
            doc.metadata["year"] = "2026"

            if "Leave" in filename:
                doc.metadata["policy_type"] = "Leave"
            elif "Attendance" in filename:
                doc.metadata["policy_type"] = "Attendance"
            elif "WFH" in filename:
                doc.metadata["policy_type"] = "Work From Home"
            elif "Reimbursement" in filename:
                doc.metadata["policy_type"] = "Reimbursement"
            elif "HR" in filename:
                doc.metadata["policy_type"] = "HR"
            elif "Employee" in filename:
                doc.metadata["policy_type"] = "Employee Guidelines"

        documents.extend(docs)

print("Documents loaded:", len(documents))

/tmp/ipykernel_2731/3526143592.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Documents loaded: 6


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 6


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS vector database created successfully!


In [6]:
question = "How many casual leaves can an employee take?"

results = vectorstore.similarity_search(
    question,
    k=3
)

for i, doc in enumerate(results):
    print("\n" + "="*60)
    print("CONTEXT", i+1)
    print("="*60)
    print(doc.page_content)
    print("\nSource:", doc.metadata["source"])
    print("Policy Type:", doc.metadata["policy_type"])


CONTEXT 1
Leave Policy

Employees are entitled to 12 casual leave days per year.
Employees are entitled to 10 sick leave days per year.
Employees are entitled to 15 annual leave days per year.
Leave requests must be submitted through the approved leave management system.
Manager approval is required before taking planned leave.
In case of emergency leave, the employee should notify the manager as soon as possible.

Source: Leave_Policy.txt
Policy Type: Leave

CONTEXT 2
Work From Home Policy

Eligible employees can work from home up to 2 days per week.
Work from home requires manager approval.
Work from home arrangements depend on business requirements.
Employees must follow normal working hours while working from home.
Employees must remain available through approved communication tools.
Employees may be required to attend office meetings, training, or team activities.

Source: WFH_Policy.txt
Policy Type: Work From Home

CONTEXT 3
Employee Guidelines

Employees should communicate prof

In [7]:
def ask_policy(question, policy_type="All", top_k=3):

    if policy_type == "All":
        results = vectorstore.similarity_search(
            question,
            k=top_k
        )
    else:
        results = vectorstore.similarity_search(
            question,
            k=top_k,
            filter={"policy_type": policy_type}
        )

    if not results:
        print("No relevant information found.")
        return

    print("\n🤖 ANSWER")
    print("-" * 50)
    print(results[0].page_content)

    print("\n📌 SOURCES")
    print("-" * 50)

    sources = []

    for doc in results:
        source = doc.metadata["source"]

        if source not in sources:
            sources.append(source)

    for source in sources:
        print("📄", source)

In [8]:
ask_policy(
    "How many sick leaves are available?",
    "Leave"
)


🤖 ANSWER
--------------------------------------------------
Leave Policy

Employees are entitled to 12 casual leave days per year.
Employees are entitled to 10 sick leave days per year.
Employees are entitled to 15 annual leave days per year.
Leave requests must be submitted through the approved leave management system.
Manager approval is required before taking planned leave.
In case of emergency leave, the employee should notify the manager as soon as possible.

📌 SOURCES
--------------------------------------------------
📄 Leave_Policy.txt


In [9]:
ask_policy("How many days can employees work from home?")


🤖 ANSWER
--------------------------------------------------
Work From Home Policy

Eligible employees can work from home up to 2 days per week.
Work from home requires manager approval.
Work from home arrangements depend on business requirements.
Employees must follow normal working hours while working from home.
Employees must remain available through approved communication tools.
Employees may be required to attend office meetings, training, or team activities.

📌 SOURCES
--------------------------------------------------
📄 WFH_Policy.txt
📄 HR_Policy.txt
📄 Leave_Policy.txt


In [10]:
ask_policy("What is the reimbursement submission period?")


🤖 ANSWER
--------------------------------------------------
Reimbursement Policy

Approved business expenses are eligible for reimbursement.
Employees must submit valid receipts for reimbursement.
Reimbursement requests should normally be submitted within 30 days.
Manager and Finance approval may be required.
Approved business travel expenses can be reimbursed.
Personal expenses are not eligible for reimbursement.

📌 SOURCES
--------------------------------------------------
📄 Reimbursement_Policy.txt
📄 Leave_Policy.txt
📄 HR_Policy.txt


In [11]:
ask_policy("What are the working hours?")


🤖 ANSWER
--------------------------------------------------
HR Policy

Working hours are from 9:00 AM to 6:00 PM, Monday to Friday.
Employees must maintain professional behavior in the workplace.
New employees have a probation period of six months.
Employee performance is reviewed annually.
Employees can contact the HR department for policy clarification.

📌 SOURCES
--------------------------------------------------
📄 HR_Policy.txt
📄 WFH_Policy.txt
📄 Leave_Policy.txt


In [12]:
ask_policy("Do I need manager approval for leave?")


🤖 ANSWER
--------------------------------------------------
Leave Policy

Employees are entitled to 12 casual leave days per year.
Employees are entitled to 10 sick leave days per year.
Employees are entitled to 15 annual leave days per year.
Leave requests must be submitted through the approved leave management system.
Manager approval is required before taking planned leave.
In case of emergency leave, the employee should notify the manager as soon as possible.

📌 SOURCES
--------------------------------------------------
📄 Leave_Policy.txt
📄 Attendance_Policy.txt
📄 Reimbursement_Policy.txt


In [13]:
ask_policy(
    "How many leaves are available?",
    policy_type="Leave"
)


🤖 ANSWER
--------------------------------------------------
Leave Policy

Employees are entitled to 12 casual leave days per year.
Employees are entitled to 10 sick leave days per year.
Employees are entitled to 15 annual leave days per year.
Leave requests must be submitted through the approved leave management system.
Manager approval is required before taking planned leave.
In case of emergency leave, the employee should notify the manager as soon as possible.

📌 SOURCES
--------------------------------------------------
📄 Leave_Policy.txt


In [ ]:
while True:

    question = input("\nAsk your policy question (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("RAG system stopped.")
        break

    ask_policy(question)


Ask your policy question (type 'exit' to stop): leave 

🤖 ANSWER
--------------------------------------------------
Leave Policy

Employees are entitled to 12 casual leave days per year.
Employees are entitled to 10 sick leave days per year.
Employees are entitled to 15 annual leave days per year.
Leave requests must be submitted through the approved leave management system.
Manager approval is required before taking planned leave.
In case of emergency leave, the employee should notify the manager as soon as possible.

📌 SOURCES
--------------------------------------------------
📄 Leave_Policy.txt
📄 Attendance_Policy.txt
📄 WFH_Policy.txt

Ask your policy question (type 'exit' to stop): work from home

🤖 ANSWER
--------------------------------------------------
Work From Home Policy

Eligible employees can work from home up to 2 days per week.
Work from home requires manager approval.
Work from home arrangements depend on business requirements.
Employees must follow normal working hour